# ============================================================
# HIGH-Throughput Virtual Screening of Kappa Opioid Receptor Inhibitors
## Author: **Babak Mamnoon**
## Docking Engine: GNINA


# ============================================================
# SECTION 1 — INSTALL REQUIRED PACKAGES
# ============================================================

In [ ]:
!pip -q install py3Dmol rdkit pandas numpy tqdm meeko

!apt-get -qq install openbabel


# ============================================================
# SECTION 2 — CHECK GPU
# ============================================================

In [ ]:
!nvidia-smi

# ============================================================
# SECTION 3 — CREATE WORKING DIRECTORY
# ============================================================

In [ ]:
import os

WORKDIR = "/content/KOR_HTVS"

os.makedirs(WORKDIR, exist_ok=True)

os.chdir(WORKDIR)

print("Current directory:", os.getcwd())

# ============================================================
# SECTION 4 — DOWNLOAD AND CLEAN THE KAPPA OPIOID RECEPTOR
# PDB ID: 4DJH
# ============================================================

In [ ]:
# Download
!wget -q https://files.rcsb.org/download/4DJH.pdb

# Clean: Keep Chain A and Ligand JDC, remove water/others
with open("4DJH.pdb", "r") as f:
    lines = f.readlines()

with open("receptor_clean.pdb", "w") as f:
    for line in lines:
        # Check ATOM and HETATM records
        if line.startswith(("ATOM", "HETATM")):
            chain = line[21].strip()
            res_name = line[17:20].strip()
            # Only keep Chain A and exclude common solvents/ions
            if chain == "A":
                if res_name not in ["HOH", "EDO", "SO4", "GOL", "CIT"]:
                    f.write(line)

        # Only write TER if it belongs to the end of a chain we are keeping
        # (In 4DJH, TER records follow the ATOM records for each chain)
        elif line.startswith("TER"):
            if line[21].strip() == "A" or line[21].strip() == "":
                f.write(line)

# Add Hydrogens for GNINA (creates receptor_prepared.pdb)
!obabel receptor_clean.pdb -O receptor_prepared.pdb -h

print("Receptor cleaned and prepared.")

# ============================================================
# SECTION 5 — VISUALIZE RECEPTOR STRUCTURE
# ============================================================

In [ ]:
import py3Dmol

# FIX: Filename must match the output from Section 4
with open("receptor_clean.pdb", "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_data, "pdb")

# 1. Set the base style for the protein (Chain A)
view.setStyle(
    {'chain': 'A'},
    {'cartoon': {'color': 'spectrum'}}
)

# 2. Add style for the ligand (JDC) so it doesn't overwrite the cartoon
view.addStyle(
    {'resn': 'JDC'},
    {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.2}}
)

view.zoomTo()
view.show()

# ============================================================
# SECTION 6 — EXTRACT CO-CRYSTALLIZED LIGAND
# ============================================================

In [ ]:
!grep "JDC" receptor_clean.pdb > reference_ligand.pdb
print("Reference ligand JDC extracted for autobox.")

# ============================================================
# SECTION 7 — VISUALIZE BINDING SITE
# ============================================================

In [ ]:
v = py3Dmol.view(width=1000, height=700)

# 1. Load cleaned receptor (Model 0)
v.addModel(open("receptor_clean.pdb").read(), "pdb")

# Style the protein
v.setStyle(
    {'model': 0, 'chain': 'A'},
    {'cartoon': {'color': 'spectrum'}}
)

# Style the NATIVE ligand (JDC)
v.addStyle(
    {'model': 0, 'resn': 'JDC'},
    {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.2}}
)

# 2. Load Docked Poses (Model 1)
# FIX: Ensure this filename matches your docking output exactly.
# If you didn't create a folder, use "test_docked.sdf" instead of "docking_results/test_docked.sdf"
try:
    v.addModelsAsFrames(open("test_docked.sdf").read(), "sdf")

    # Style the docked poses (Cyan)
    v.setStyle(
        {'model': 1},
        {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.2}}
    )
except FileNotFoundError:
    print("Error: 'test_docked.sdf' not found. Check your docking output filename.")

# 3. Focus on the binding site
v.zoomTo({'resn': 'JDC'})

v.show()

# ============================================================
# SECTION 8 — INSTALL GNINA
# ============================================================

In [ ]:
!wget -q https://github.com/gnina/gnina/releases/download/v1.0.3/gnina

!chmod +x gnina

!./gnina --version

# ============================================================
# SECTION 9 — FETCH LIGAND LIBRARY FROM GITHUB
# ============================================================

In [ ]:
# Replace with your raw GitHub CSV URL
GITHUB_CSV_URL = "https://raw.githubusercontent.com/Babakmamnoon/High-Throughput-Virtual-Screening-of-Human-Kappa-Opioid-Receptor/refs/heads/main/ligands.csv"

!wget -O ligands.csv $GITHUB_CSV_URL

import pandas as pd

df = pd.read_csv("ligands.csv")

print(df.head())

# ============================================================
# SECTION 10 — EXPECTED CSV FORMAT
# ============================================================
"""
Expected CSV columns:

compound_id,smiles

Example:

compound_1,CCO
compound_2,CCN(CC)CC
"""

In [ ]:
import pandas as pd

# Load original ChEMBL dataset, specifying the semicolon delimiter
# and handling potential quotes
chembl_df = pd.read_csv("ligands.csv", sep=';', quotechar='"', skipinitialspace=True)

print("Original dataset shape:")
print(chembl_df.shape)

print("\nColumns in dataset:")
print(chembl_df.columns.tolist())

# ============================================================
# Select Required Columns
# ============================================================

"""
Typical ChEMBL columns:

- molecule_chembl_id (now 'Compound ChEMBL ID')
- canonical_smiles (now 'Smiles')

We rename them to:

- compound_id
- smiles
"""

clean_df = chembl_df[
    [
        "Compound ChEMBL ID", # Corrected column name
        "Smiles"            # Corrected column name
    ]
].copy()

# Rename columns
clean_df.columns = [
    "compound_id",
    "smiles"
]

# ============================================================
# Remove Missing Values
# ============================================================

clean_df = clean_df.dropna(subset=["smiles"])

# ============================================================
# Remove Duplicate SMILES
# ============================================================

clean_df = clean_df.drop_duplicates(
    subset=["smiles"]
).reset_index(drop=True)

print("\nClean dataset shape:")
print(clean_df.shape)

# ============================================================
# Save Clean Dataset
# ============================================================

clean_df.to_csv(
    "clean_ligands.csv",
    index=False
)

print("\nClean ligand file saved.")

# Replace df with cleaned dataset
df = clean_df

df.head()

# ============================================================
# SECTION 11 — LIGAND STANDARDIZATION
# ============================================================

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm

os.makedirs("ligands_sdf", exist_ok=True)

valid_compounds = []

for idx, row in tqdm(clean_df.iterrows(), total=len(clean_df)):

    compound_id = str(row["compound_id"])
    smiles = str(row["smiles"])

    try:
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            continue

        # Add hydrogens
        mol = Chem.AddHs(mol)

        # Generate 3D coordinates
        AllChem.EmbedMolecule(
            mol,
            randomSeed=42
        )

        # MMFF optimization
        AllChem.MMFFOptimizeMolecule(mol)

        output_file = f"ligands_sdf/{compound_id}.sdf"

        writer = Chem.SDWriter(output_file)
        writer.write(mol)
        writer.close()

        valid_compounds.append(compound_id)

    except:
        continue

print(f"Valid ligands prepared: {len(valid_compounds)}")

# ============================================================
# SECTION 12 — CONVERT LIGANDS TO PDBQT-LIKE FORMAT
# ============================================================

In [ ]:
os.makedirs("ligands_prepared", exist_ok=True)

sdf_files = os.listdir("ligands_sdf")

for sdf in tqdm(sdf_files):

    input_path = f"ligands_sdf/{sdf}"

    output_path = f"ligands_prepared/{sdf.replace('.sdf','.pdbqt')}"

    !obabel {input_path} -O {output_path} -h

print("Ligands converted successfully.")

# ============================================================
# SECTION 13 — TEST SINGLE DOCKING
# ============================================================

In [ ]:
test_ligand = os.listdir("ligands_sdf")[0]

!./gnina \
-r receptor_prepared.pdb \
-l ligands_sdf/{test_ligand} \
--autobox_ligand reference_ligand.pdb \
--autobox_add 6 \
-o test_docked.sdf \
--exhaustiveness 8 \
--cnn_scoring rescore \
--seed 42

print("Single docking test completed.")

# ============================================================
# SECTION 14 — HIGH-THROUGHPUT VIRTUAL SCREENING
# ============================================================

In [ ]:
import glob
import subprocess
import os
from tqdm import tqdm

# Create the directory where results will be stored
os.makedirs("docking_results", exist_ok=True)

# Find all ligand files in the ligands_sdf folder
ligand_files = glob.glob("ligands_sdf/*.sdf")

print(f"Found {len(ligand_files)} ligands. Starting HTVS...")

print(f"Current Directory: {os.getcwd()}")
print(f"Gnina exists here: {os.path.exists('./gnina')}")
print(f"Receptor exists here: {os.path.exists('receptor_prepared.pdb')}")

for ligand in tqdm(ligand_files):
    ligand_name = os.path.basename(ligand).replace(".sdf","")
    output_file = f"docking_results/{ligand_name}_docked.sdf"

    # Skip if already docked (useful for resuming interrupted jobs)
    if os.path.exists(output_file):
        continue

    # GNINA Command
    # 1. Changed -r to receptor_prepared.pdb (Hydrogenated version)
    # 2. Changed --autobox_ligand to receptor_clean.pdb (Contains native JDC)
    command = f"""
    ./gnina \
    -r receptor_prepared.pdb \
    -l {ligand} \
    --autobox_ligand receptor_clean.pdb \
    --autobox_add 6 \
    --exhaustiveness 8 \
    --cnn_scoring rescore \
    --seed 42 \
    -o {output_file} \
    --num_modes 1
    """

    # Use check=True to catch errors if gnina fails
    result = subprocess.run(command, shell=True, capture_output=True, text=True)

    if result.returncode != 0:
      print(f"Error docking {ligand_name}: {result.stderr}")

print("HTVS completed.")

# ============================================================
# SECTION 15 — EXTRACT GNINA SCORES
# ============================================================

In [ ]:
import re
import glob
import os
import pandas as pd

docking_summary = []

# Ensure we are looking in the correct results folder
result_files = glob.glob("docking_results/*_docked.sdf")

for file in result_files:
    ligand_name = os.path.basename(file).replace("_docked.sdf","")

    with open(file, "r") as f:
        content = f.read()

    # Regex for Vina-style minimized affinity (Lower is better)
    affinity_match = re.search(r'> <minimizedAffinity>\s*\n([-.\d]+)', content)

    # Regex for CNN affinity (Higher is usually better)
    cnn_match = re.search(r'> <CNNaffinity>\s*\n([-.\d]+)', content)

    # Convert to float for proper numerical sorting later
    try:
        vina_score = float(affinity_match.group(1)) if affinity_match else None
        cnn_score = float(cnn_match.group(1)) if cnn_match else None
    except (ValueError, TypeError):
        vina_score, cnn_score = None, None

    docking_summary.append([
        ligand_name,
        vina_score,
        cnn_score
    ])

# Create DataFrame
results_df = pd.DataFrame(
    docking_summary,
    columns=["Ligand", "Vina_Affinity", "CNN_Affinity"]
)

# Drop any rows where docking failed (None values)
results_df = results_df.dropna(subset=["Vina_Affinity", "CNN_Affinity"])

# SORTING:
# For Vina_Affinity, lower (more negative) is better.
# For CNN_Affinity, higher is typically better.
# Here we sort by CNN_Affinity (Top scores at the top)
results_df = results_df.sort_values(by="CNN_Affinity", ascending=False)

# Save to CSV
results_df.to_csv("KOR_HTVS_Results.csv", index=False)

# Display top 20
print(f"Successfully processed {len(results_df)} ligands.")
results_df.head(20)

# ============================================================
# SECTION 16 — VISUALIZE TOP HIT
# ============================================================

In [ ]:
# Get the ligand name of the top scorer
top_hit = results_df.iloc[0]["Ligand"]
top_hit_file = f"docking_results/{top_hit}_docked.sdf"

v = py3Dmol.view(width=1000, height=700)

# 1. Load Receptor (Model 0)
v.addModel(open("receptor_clean.pdb").read(), "pdb")
v.setStyle({'model': 0}, {'cartoon': {'color': 'white', 'opacity': 0.8}})

# 2. Load Top Hit Poses (Model 1)
# We use 'sdf' to ensure the format is explicitly recognized
v.addModelsAsFrames(open(top_hit_file).read(), "sdf")

# Style the docked ligand
v.setStyle(
    {'model': 1},
    {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.2}}
)

# 3. Optional: Style the reference ligand JDC if you want to compare
# v.addStyle({'model': 0, 'resn': 'JDC'}, {'stick': {'colorscheme': 'greenCarbon', 'opacity': 0.5}})

# 4. Focus on the Top Hit
v.zoomTo({'model': 1})

v.show()
print(f"Visualizing Top Hit: {top_hit}")

# ============================================================
# SECTION 17 — VISUALIZE OTHER HITS
# ============================================================

In [ ]:
import py3Dmol
import pandas as pd

# Sort ligands by CNN score (highest = best)
results_sorted = results_df.sort_values(
    by="CNN_Affinity",
    ascending=False
).reset_index(drop=True)

# Print ranking
print("Ligands ranked by CNN_Affinity:\n")
print(results_sorted[["Ligand", "CNN_Affinity", "Vina_Affinity"]])

# Visualize top 5 ligands one by one
for i in range(5):

    ligand_name = results_sorted.iloc[i]["Ligand"]
    cnn_score = results_sorted.iloc[i]["CNN_Affinity"]
    vina_score = results_sorted.iloc[i]["Vina_Affinity"]

    ligand_file = f"docking_results/{ligand_name}_docked.sdf"

    print(f"\nVisualizing Rank #{i+1}")
    print(f"Ligand: {ligand_name}")
    print(f"CNN_Affinity: {cnn_score}")
    print(f"Vina_Affinity: {vina_score}")

    v = py3Dmol.view(width=1000, height=700)

    # =========================
    # 1. Load receptor
    # =========================
    v.addModel(open("receptor_clean.pdb").read(), "pdb")

    v.setStyle(
        {'model': 0},
        {'cartoon': {'color': 'white', 'opacity': 0.8}}
    )

    # =========================
    # 2. Load docked ligand
    # =========================
    v.addModelsAsFrames(open(ligand_file).read(), "sdf")

    v.setStyle(
        {'model': 1},
        {'stick': {
            'colorscheme': 'cyanCarbon',
            'radius': 0.2
        }}
    )

    # =========================
    # 3. Optional reference ligand
    # =========================
    # v.addStyle(
    #     {'model': 0, 'resn': 'JDC'},
    #     {'stick': {'colorscheme': 'greenCarbon'}}
    # )

    # =========================
    # 4. Zoom to ligand
    # =========================
    v.zoomTo({'model': 1})

    # =========================
    # 5. Show visualization
    # =========================
    v.show()

# ============================================================
# SECTION 18 — SAVE COMPLETE PROJECT
# ============================================================

In [ ]:
!zip -r KOR_HTVS_Project.zip \
receptor_prepared.pdb \
docking_results \
KOR_HTVS_Results.csv \
ligands_sdf

print("Project archived successfully.")

# ============================================================
# SECTION 19 — DOWNLOAD RESULTS
# ============================================================

In [ ]:
from google.colab import files

files.download("KOR_HTVS_Project.zip")